In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round
from pyspark.sql.functions import from_unixtime, to_date, format_number

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType

In [ ]:
import os

# Path to Hadoop/YARN config (adjust to your system if needed, /etc/hadoop/conf is standard)
os.environ["HADOOP_CONF_DIR"] = "/etc/hadoop/conf"
os.environ["YARN_CONF_DIR"] = "/etc/hadoop/conf"

print("HADOOP_CONF_DIR:", os.environ.get("HADOOP_CONF_DIR"))
print("YARN_CONF_DIR:", os.environ.get("YARN_CONF_DIR"))

HADOOP_CONF_DIR: /etc/hadoop/conf
YARN_CONF_DIR: /etc/hadoop/conf


In [ ]:
# -----------------------------------
# 1. Initialize Spark
# -----------------------------------
spark = SparkSession.builder \
    .appName("BikeStore HDFS to Hive ETL") \
    .master("yarn") \
    .config("spark.submit.deployMode", "client") \
    .enableHiveSupport() \
    .getOrCreate()

26/04/25 04:15:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/25 04:15:43 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [ ]:
print(spark.sparkContext.master)

yarn


# Load

In [ ]:
# -----------------------------------
# 2. Read Parquet Files From HDFS
# -----------------------------------
print("Loading Sales Data...")
df_customers   = spark.read.parquet("/staging_area/Sales/Customer")
df_orders      = spark.read.parquet("/staging_area/Sales/Orders")
df_order_items = spark.read.parquet("/staging_area/Sales/Order_Items")
df_stores      = spark.read.parquet("/staging_area/Sales/Stores")
df_staffs      = spark.read.parquet("/staging_area/Sales/Staffs")

print("Loading Production Data...")
df_brands     = spark.read.parquet("/staging_area/Production/Brands")
df_categories = spark.read.parquet("/staging_area/Production/Categories")
df_products   = spark.read.parquet("/staging_area/Production/Products")
df_stocks     = spark.read.parquet("/staging_area/Production/Stocks")

Loading Sales Data...


26/04/25 04:18:45 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/25 04:19:00 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/25 04:19:15 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


Loading Production Data...


In [ ]:
# Testing

tables = {
    "customers"  : df_customers,
    "orders"     : df_orders,
    "order_items": df_order_items,
    "stores"     : df_stores,
    "staffs"     : df_staffs,
    "brands"     : df_brands,
    "categories" : df_categories,
    "products"   : df_products,
    "stocks"     : df_stocks,
}

for name, df in tables.items():
    print(f"  {name:<15} → {df.count():>5} rows  |  {len(df.columns)} columns")

  customers       →  1445 rows  |  9 columns


  orders          →  1615 rows  |  8 columns


  order_items     →  4722 rows  |  6 columns


  stores          →     3 rows  |  8 columns


  staffs          →    10 rows  |  8 columns


  brands          →     9 rows  |  2 columns


  categories      →     7 rows  |  2 columns


  products        →   321 rows  |  6 columns


  stocks          →   939 rows  |  3 columns


In [ ]:
#Inspecting Data

for name, df in tables.items():
    print(f"\n── {name} ──")
    df.show(2)
    df.printSchema()


── customers ──


+-----------+----------+---------+-----+--------------------+-----------------+------------+-----+--------+
|customer_id|first_name|last_name|phone|               email|           street|        city|state|zip_code|
+-----------+----------+---------+-----+--------------------+-----------------+------------+-----+--------+
|
|
+-----------+----------+---------+-----+--------------------+-----------------+------------+-----+--------+
only showing top 2 rows

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)


── orders ──


+--------+-----------+------------+-------------+-------------+-------------+--------+--------+
|order_id|customer_id|order_status|   order_date|required_date| shipped_date|store_id|staff_id|
+--------+-----------+------------+-------------+-------------+-------------+--------+--------+
|       1|        259|           4|1451574000000|1451746800000|1451746800000|       1|       2|
|       2|       1212|           4|1451574000000|1451833200000|1451746800000|       2|       6|
+--------+-----------+------------+-------------+-------------+-------------+--------+--------+
only showing top 2 rows

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: integer (nullable = true)
 |-- order_date: long (nullable = true)
 |-- required_date: long (nullable = true)
 |-- shipped_date: long (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- staff_id: integer (nullable = true)


── order_items ──


+--------+-------+----------+--------+----------+--------+
|order_id|item_id|product_id|quantity|list_price|discount|
+--------+-------+----------+--------+----------+--------+
|       1|      1|        20|       1|    599.99|    0.20|
|       1|      2|         8|       2|   1799.99|    0.07|
+--------+-------+----------+--------+----------+--------+
only showing top 2 rows

root
 |-- order_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- list_price: string (nullable = true)
 |-- discount: string (nullable = true)


── stores ──


+--------+----------------+--------------+--------------------+------------------+----------+-----+--------+
|store_id|      store_name|         phone|               email|            street|      city|state|zip_code|
+--------+----------------+--------------+--------------------+------------------+----------+-----+--------+
|
|
+--------+----------------+--------------+--------------------+------------------+----------+-----+--------+
only showing top 2 rows

root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)


── staffs ──


+--------+----------+---------+--------------------+--------------+------+--------+----------+
|staff_id|first_name|last_name|               email|         phone|active|store_id|manager_id|
+--------+----------+---------+--------------------+--------------+------+--------+----------+
|       1|   Fabiola|  Jackson|fabiola.jackson@b...|(831) 555-5554|     1|       1|         0|
|       2|    Mireya| Copeland|mireya.copeland@b...|(831) 555-5555|     1|       1|         1|
+--------+----------+---------+--------------------+--------------+------+--------+----------+
only showing top 2 rows

root
 |-- staff_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- active: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- manager_id: integer (nullable = true)


── brands ──


+--------+----------+
|brand_id|brand_name|
+--------+----------+
|
|
+--------+----------+
only showing top 2 rows

root
 |-- brand_id: integer (nullable = true)
 |-- brand_name: string (nullable = true)


── categories ──


+-----------+------------------+
|category_id|     category_name|
+-----------+------------------+
|
|
+-----------+------------------+
only showing top 2 rows

root
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)


── products ──


+----------+--------------------+--------+-----------+----------+----------+
|product_id|        product_name|brand_id|category_id|model_year|list_price|
+----------+--------------------+--------+-----------+----------+----------+
|         1|     Trek 820 - 2016|       9|          6|      2016|    379.99|
|         2|Ritchey Timberwol...|       5|          6|      2016|    749.99|
+----------+--------------------+--------+-----------+----------+----------+
only showing top 2 rows

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- list_price: string (nullable = true)


── stocks ──


+--------+----------+--------+
|store_id|product_id|quantity|
+--------+----------+--------+
|       1|         1|      27|
|       1|         2|       5|
+--------+----------+--------+
only showing top 2 rows

root
 |-- store_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)



After inspecting the data, we found the following issues:
    -> Issue 1:  Dates stored as Unix timestamps (milliseconds),
                 therefore will need to convert to a date type
    -> Issue 2: list_price and discount are strings, not decimals

# Cleaning

In [ ]:
# ── df_orders: convert Unix ms timestamps → dates ──────────────
df_orders = df_orders \
    .withColumn("order_date",    F.to_date(F.from_unixtime(F.col("order_date")    / 1000))) \
    .withColumn("required_date", F.to_date(F.from_unixtime(F.col("required_date") / 1000))) \
    .withColumn("shipped_date",  F.to_date(F.from_unixtime(F.col("shipped_date")  / 1000)))

# ── df_order_items: cast price, discount ───────────────────────
df_order_items = df_order_items \
    .withColumn("list_price", F.col("list_price").cast(DecimalType(10, 2))) \
    .withColumn("discount",   F.col("discount").cast(DecimalType(4, 2)))

# ── df_products: cast price ─────────────────────────────────────
df_products = df_products \
    .withColumn("list_price", F.col("list_price").cast(DecimalType(10, 2)))

In [ ]:
print("── orders sample after fix ──")
df_orders.show(2)
df_orders.printSchema()

print("\n── order_items schema ──")
df_order_items.printSchema()

print("\n── products schema ──")
df_products.printSchema()

── orders sample after fix ──


+--------+-----------+------------+----------+-------------+------------+--------+--------+
|order_id|customer_id|order_status|order_date|required_date|shipped_date|store_id|staff_id|
+--------+-----------+------------+----------+-------------+------------+--------+--------+
|       1|        259|           4|2016-01-01|   2016-01-03|  2016-01-03|       1|       2|
|       2|       1212|           4|2016-01-01|   2016-01-04|  2016-01-03|       2|       6|
+--------+-----------+------------+----------+-------------+------------+--------+--------+
only showing top 2 rows

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- staff_id: integer (nullable = true)


── order_items schema ──
root
 |-- order_id: integer (nullable = true)
 |-- ite

# Joining tables to make fact table
Join all 9 tables into one flat fact table.

Only **completed orders** (order_status = 4) are included.

    line_revenue = quantity × list_price × (1 − discount)
    list_price here = the actual selling price from order_items  
    catalog_price = the product's standard price from products

In [ ]:
# ── Filter completed orders only ─────────────────────────────
completed_orders = df_orders.filter(F.col("order_status") == 4)
print(f"Completed orders : {completed_orders.count()} / {df_orders.count()} total")

# ── Rename products list_price to catalog_price to avoid ambiguity since both df_products AND df_order_items have a column called list_price. ──
df_products_renamed = df_products.withColumnRenamed("list_price", "catalog_price")

# ── Compute line-level revenue ───────────────────────────────
df_order_items = df_order_items.withColumn(
    "line_revenue",
    F.col("quantity") * F.col("list_price") * (1 - F.col("discount"))
)

# ── Join everything ──────────────────────────────────────────
fact = completed_orders \
    .join(df_order_items, "order_id") \
    .join(df_stores.select(
        "store_id", "store_name",
        F.col("city"), F.col("state")
    ), "store_id") \
    .join(
        df_staffs.select(
            F.col("staff_id"),
            F.concat_ws(" ", F.col("first_name"), F.col("last_name")).alias("staff_name")
        ), "staff_id"
    ) \
    .join(df_products_renamed, "product_id") \
    .join(df_brands, "brand_id") \
    .join(df_categories, "category_id") \
    .join(
        df_customers.select(
            F.col("customer_id"),
            F.concat_ws(" ", F.col("first_name"), F.col("last_name")).alias("customer_name"),
            F.col("city").alias("customer_city"),
            F.col("state").alias("customer_state")
        ), "customer_id"
    )

# ── Drop rows with null/zero revenue ────────────────────────
fact = fact.filter(
    F.col("order_date").isNotNull() &
    F.col("line_revenue").isNotNull() &
    (F.col("line_revenue") > 0)
)

# ── Cache fact table ─────────────────────────────────────────
fact.cache()
fact_count = fact.count()
print(f"\n Fact table built and cached: {fact_count} rows")
print(f"   Columns: {len(fact.columns)}")
fact.printSchema()

Completed orders : 1445 / 1615 total


26/04/25 04:27:28 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



 Fact table built and cached: 4214 rows
   Columns: 28
root
 |-- customer_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- brand_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_status: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- list_price: decimal(10,2) (nullable = true)
 |-- discount: decimal(4,2) (nullable = true)
 |-- line_revenue: decimal(27,4) (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- staff_name: string (nullable = false)
 |-- product_name: string (nullable = true)
 |-- model_year: integer (nullable = true)
 |

In [ ]:
fact.select("brand_id", "brand_name").show(10, False)

+--------+----------+
|brand_id|brand_name|
+--------+----------+
     |
  |
    |
     |
  |
  |
  |
  |
    |
  |
+--------+----------+
only showing top 10 rows



In [ ]:
# Clean the brand_name column by replacing \r and \n with an empty string
fact = fact.withColumn(
    "brand_name",
    F.regexp_replace(F.col("brand_name"), r"[\r\n]", "")
)

In [ ]:
fact.select("brand_id", "brand_name").show(10, False)

+--------+----------+
|brand_id|brand_name|
+--------+----------+
|9       |Trek      |
|1       |Electra   |
|8       |Surly     |
|9       |Trek      |
|1       |Electra   |
|1       |Electra   |
|1       |Electra   |
|1       |Electra   |
|8       |Surly     |
|5       |Ritchey   |
+--------+----------+
only showing top 10 rows



In [ ]:
from pyspark.sql.types import StringType

def strip_hidden_chars(df):
    """
    Finds all String columns in a DataFrame and removes hidden
    carriage returns (\r) and newlines (\n).
    """
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(
                field.name,
                F.regexp_replace(F.col(field.name), r"[\r\n]", "")
            )
    return df

# Clean the fact table
fact = strip_hidden_chars(fact)


# Analysis

In [ ]:
#1. Revenue by Store
revenue_by_store = fact \
    .groupBy("store_id", "store_name", "city", "state") \
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
        F.sum("quantity").alias("total_units_sold"),
        F.countDistinct("order_id").alias("total_orders")
    ) \
    .orderBy(F.desc("total_revenue"))

revenue_by_store.show()
revenue_by_store.write.mode("overwrite").parquet("/staging_area/Analysis/revenue_by_store")


+--------+----------------+----------+-----+-------------+----------------+------------+
|store_id|      store_name|      city|state|total_revenue|total_units_sold|total_orders|
+--------+----------------+----------+-----+-------------+----------------+------------+
|       2|   Baldwin Bikes|   Baldwin|   NY|   4701209.57|            4427|        1019|
|       1|Santa Cruz Bikes|Santa Cruz|   CA|   1255491.65|            1236|         284|
|       3|   Rowlett Bikes|   Rowlett|   TX|    705914.03|             655|         142|
+--------+----------------+----------+-----+-------------+----------------+------------+



In [ ]:
#2. Revenue by Brand
revenue_by_brand = fact \
    .groupBy("brand_id", "brand_name") \
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
        F.sum("quantity").alias("total_units_sold"),
        F.countDistinct("order_id").alias("total_orders")
    ) \
    .orderBy(F.desc("total_revenue"))

revenue_by_brand.show()
revenue_by_brand.write.mode("overwrite").parquet("/staging_area/Analysis/revenue_by_brand")


+--------+------------+-------------+----------------+------------+
|brand_id|  brand_name|total_revenue|total_units_sold|total_orders|
+--------+------------+-------------+----------------+------------+
|       9|        Trek|   3919362.61|            1564|         766|
|       1|     Electra|   1019610.57|            2329|         980|
|       8|       Surly|    868807.25|             838|         490|
|       7|Sun Bicycles|    310418.23|             670|         355|
|       2|        Haro|    163783.26|             297|         175|
|       3|      Heller|    157219.74|             130|          91|
|       4| Pure Cycles|    143213.55|             360|         221|
|       5|     Ritchey|     78186.46|             117|          76|
|       6|     Strider|      2013.59|              13|           9|
+--------+------------+-------------+----------------+------------+



In [ ]:
#3. Revenue by Category
revenue_by_category = fact \
    .groupBy("category_id", "category_name") \
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
        F.sum("quantity").alias("total_units_sold"),
        F.countDistinct("order_id").alias("total_orders")
    ) \
    .orderBy(F.desc("total_revenue"))

revenue_by_category.show(truncate=False)
revenue_by_category.write.mode("overwrite").parquet("/staging_area/Analysis/revenue_by_category")


+-----------+-------------------+-------------+----------------+------------+
|category_id|category_name      |total_revenue|total_units_sold|total_orders|
+-----------+-------------------+-------------+----------------+------------+
|6          |Mountain Bikes     |2486420.26   |1607            |788         |
|7          |Road Bikes         |1327155.64   |449             |251         |
|3          |Cruisers Bicycles  |866524.40    |1865            |865         |
|5          |Electric Bikes     |733493.75    |252             |164         |
|4          |Cyclocross Bicycles|642584.83    |365             |228         |
|2          |Comfort Bicycles   |346450.00    |733             |425         |
|1          |Children Bicycles  |259986.37    |1047            |563         |
+-----------+-------------------+-------------+----------------+------------+



In [ ]:
#4. Top Customers
top_customers = fact \
    .groupBy("customer_id", "customer_name", "customer_city", "customer_state") \
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("lifetime_value"),
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_units_bought"),
        F.max("order_date").alias("last_order_date")
    ) \
    .orderBy(F.desc("lifetime_value"))

top_customers.show(10, truncate=False)
top_customers.write.mode("overwrite").parquet("/staging_area/Analysis/top_customers")


+-----------+----------------+-------------+--------------+--------------+------------+------------------+---------------+
|customer_id|customer_name   |customer_city|customer_state|lifetime_value|total_orders|total_units_bought|last_order_date|
+-----------+----------------+-------------+--------------+--------------+------------+------------------+---------------+
|73         |Melanie Hayes   |Liverpool    |NY            |27050.72      |1           |9                 |2017-06-11     |
|122        |Shena Carter    |Howard Beach |NY            |24890.62      |1           |5                 |2018-01-25     |
|1224       |Abram Copeland  |Harlingen    |TX            |24607.03      |1           |8                 |2017-06-05     |
|1214       |Brigid Sharp    |Santa Clara  |CA            |20648.95      |1           |5                 |2018-01-15     |
|425        |Augustina Joyner|Mount Vernon |NY            |20509.43      |1           |8                 |2018-01-07     |
|238        |Cin

In [ ]:
#5. Staff Performance
staff_performance = fact \
    .groupBy("staff_id", "staff_name", "store_name") \
    .agg(
        F.countDistinct("order_id").alias("completed_orders"),
        F.round(F.sum("line_revenue"), 2).alias("total_revenue_generated"),
        F.countDistinct("customer_id").alias("unique_customers_served")
    ) \
    .orderBy(F.desc("completed_orders"))

staff_performance.show(truncate=False)
staff_performance.write.mode("overwrite").parquet("/staging_area/Analysis/staff_performance")


+--------+---------------+----------------+----------------+-----------------------+-----------------------+
|staff_id|staff_name     |store_name      |completed_orders|total_revenue_generated|unique_customers_served|
+--------+---------------+----------------+----------------+-----------------------+-----------------------+
|6       |Marcelene Boyer|Baldwin Bikes   |521             |2405217.85             |521                    |
|7       |Venita Daniel  |Baldwin Bikes   |498             |2295991.72             |498                    |
|3       |Genna Serrano  |Santa Cruz Bikes|147             |643627.98              |147                    |
|2       |Mireya Copeland|Santa Cruz Bikes|137             |611863.67              |137                    |
|9       |Layla Terrell  |Rowlett Bikes   |71              |330449.23              |71                     |
|8       |Kali Vargas    |Rowlett Bikes   |71              |375464.80              |71                     |
+--------+---------

In [ ]:
#6. Monthly Revenue Trend
monthly_revenue = fact \
    .withColumn("year",  F.year("order_date")) \
    .withColumn("month", F.month("order_date")) \
    .groupBy("year", "month") \
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_units_sold")
    ) \
    .orderBy("year", "month")

monthly_revenue.show(36, truncate=False)
monthly_revenue.write.mode("overwrite").parquet("/staging_area/Analysis/monthly_revenue")


+----+-----+-------------+------------+----------------+
|year|month|total_revenue|total_orders|total_units_sold|
+----+-----+-------------+------------+----------------+
|2016|1    |215146.42    |50          |221             |
|2016|2    |148506.36    |47          |215             |
|2016|3    |180600.33    |55          |213             |
|2016|4    |164399.08    |41          |173             |
|2016|5    |197179.09    |49          |215             |
|2016|6    |205463.98    |44          |192             |
|2016|7    |199556.81    |50          |211             |
|2016|8    |212827.11    |59          |234             |
|2016|9    |271356.68    |66          |279             |
|2016|10   |212078.08    |64          |254             |
|2016|11   |176050.57    |42          |175             |
|2016|12   |189684.69    |53          |220             |
|2017|1    |285616.48    |50          |229             |
|2017|2    |281968.30    |54          |250             |
|2017|3    |308157.42    |66   

In [ ]:
#7. Product Summary
product_summary = fact \
    .groupBy("product_id", "product_name", "brand_name", "category_name", "model_year") \
    .agg(
        F.sum("quantity").alias("total_units_sold"),
        F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("times_ordered"),
        F.round(F.avg("catalog_price"), 2).alias("avg_catalog_price"),
        F.round(F.avg("list_price"), 2).alias("avg_selling_price")
    ) \
    .orderBy(F.desc("total_revenue"))

product_summary.show(10, truncate=False)
product_summary.write.mode("overwrite").parquet("/staging_area/Analysis/product_summary")


+----------+-------------------------------------+----------+-------------------+----------+----------------+-------------+-------------+-----------------+-----------------+
|product_id|product_name                         |brand_name|category_name      |model_year|total_units_sold|total_revenue|times_ordered|avg_catalog_price|avg_selling_price|
+----------+-------------------------------------+----------+-------------------+----------+----------------+-------------+-------------+-----------------+-----------------+
|7         |Trek Slash 8 27.5 - 2016             |Trek      |Mountain Bikes     |2016      |151             |544318.64    |99           |3999.99          |3999.99          |
|9         |Trek Conduit+ - 2016                 |Trek      |Electric Bikes     |2016      |142             |381148.73    |98           |2999.99          |2999.99          |
|4         |Trek Fuel EX 8 29 - 2016             |Trek      |Mountain Bikes     |2016      |138             |354813.78    |94     

In [ ]:
#8. Low Stock Alert
low_stock = df_stocks \
    .join(
        df_products_renamed.select(
            "product_id", "product_name", "brand_id", "category_id", "catalog_price"
        ),
        "product_id"
    ) \
    .join(df_brands, "brand_id") \
    .join(df_categories, "category_id") \
    .join(df_stores.select("store_id", "store_name"), "store_id") \
    .filter(F.col("quantity") < 5) \
    .select(
        "store_id", "store_name",
        "product_id", "product_name",
        "brand_name", "category_name",
        "quantity",
        F.col("catalog_price").alias("list_price")
    ) \
    .orderBy("store_name", "quantity")

# Clean the low_stock table
low_stock = strip_hidden_chars(low_stock)

low_stock.show(20, truncate=False)
low_stock.write.mode("overwrite").parquet("/staging_area/Analysis/low_stock_alert")


+--------+-------------+----------+---------------------------------------------+------------+-----------------+--------+----------+
|store_id|store_name   |product_id|product_name                                 |brand_name  |category_name    |quantity|list_price|
+--------+-------------+----------+---------------------------------------------+------------+-----------------+--------+----------+
|2       |Baldwin Bikes|158       |Trek CrossRip 1 - 2018                       |Trek        |Road Bikes       |0       |959.99    |
|2       |Baldwin Bikes|22        |Electra Girl's Hawaii 1 (16-inch) - 2015/2016|Electra     |Children Bicycles|0       |269.99    |
|2       |Baldwin Bikes|175       |Trek Domane SLR Frameset - 2018              |Trek        |Road Bikes       |0       |3199.99   |
|2       |Baldwin Bikes|91        |Trek Precaliber 24 (21-Speed) - Girls - 2017 |Trek        |Children Bicycles|0       |349.99    |
|2       |Baldwin Bikes|184       |Trek Domane SL 6 Disc - 2018      

In [ ]:
# Verify All 8 Analysis Outputs
analysis_tables = [
    "revenue_by_store",
    "revenue_by_brand",
    "revenue_by_category",
    "top_customers",
    "staff_performance",
    "monthly_revenue",
    "product_summary",
    "low_stock_alert",
]

print("=" * 60)
print(f"  {'Table':<30} {'Rows':>8}  {'Cols':>5}  Status")
print("=" * 60)
for table in analysis_tables:
    path = f"/staging_area/Analysis/{table}"
    try:
        df = spark.read.parquet(path)
        print(f"  {table:<30} {df.count():>8}  {len(df.columns):>5}  Completed")
    except Exception as e:
        print(f"  {table:<30} {'—':>8}  {'—':>5}  Not Completed {e}")
print("=" * 60)


  Table                              Rows   Cols  Status


  revenue_by_store                      3      7  Completed


  revenue_by_brand                      9      5  Completed


  revenue_by_category                   7      5  Completed


  top_customers                      1445      8  Completed


  staff_performance                     6      6  Completed


  monthly_revenue                      27      5  Completed


  product_summary                     278     10  Completed


  low_stock_alert                     166      8  Completed


26/04/25 16:30:10 WARN HeartbeatReceiver: Removing executor 2 with no recent heartbeats: 39595670 ms exceeds timeout 120000 ms
26/04/25 16:30:41 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 39599774 ms exceeds timeout 120000 ms
26/04/25 16:32:01 ERROR YarnScheduler: Lost executor 2 on localhost: Executor heartbeat timed out after 39595670 ms
26/04/25 16:32:09 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_167_0 !
26/04/25 16:32:22 ERROR YarnScheduler: Lost executor 1 on localhost: Executor heartbeat timed out after 39599774 ms


In [ ]:
# Release cache
fact.unpersist()
print("\n Fact table cache released.")